In [ ]:
import pandas as pd
import numpy as np

In [4]:
cols_needed = ["order",	"species", "speciesKey", "isInvasive", "decimalLatitude", "decimalLongitude",
               "eventDate", "day", "eventTime", "elevation", "basisOfRecord", "coordinateUncertaintyInMeters", "hasCoordinate", "hasGeospatialIssues", "iucnRedListCategory"]
dtypes = {"decimalLatitude": "float32", "decimalLongitude": "float32"}

chunks = []
for chunk in pd.read_csv('/content/drive/MyDrive/occurrence.txt', sep='\t',
                          usecols=cols_needed, dtype=dtypes,
                          chunksize=200_000, low_memory=False):
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(df.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/occurrence.txt'

In [ ]:
df.head()

,basisOfRecord,eventDate,eventTime,day,decimalLatitude,decimalLongitude,coordinateUncertaintyInMeters,order,elevation,hasCoordinate,hasGeospatialIssues,speciesKey,species,isInvasive,iucnRedListCategory
0,HUMAN_OBSERVATION,2024-01-08T09:41:10,09:41:10-05:00,8.0,46.232197,-67.793869,34.0,Hemiptera,NaN,True,false,6PGH7,Leptoglossus phyllopus,NaN,NaN
1,HUMAN_OBSERVATION,2024-01-24T17:54,17:54:00-05:00,24.0,43.022518,-77.060684,15.0,Lepidoptera,NaN,True,false,943K8,Hypena scabra,NaN,NaN
2,HUMAN_OBSERVATION,2024-02-09T08:50,08:50:00-05:00,9.0,42.450188,-71.120193,6.0,Coleoptera,NaN,True,false,8NBZS,Photinus corruscus,NaN,LC
3,HUMAN_OBSERVATION,2024-02-27T14:06:18,14:06:18-05:00,27.0,42.445595,-72.680771,1111.0,Coleoptera,NaN,True,false,XTZS,Conotrachelus carinifer,NaN,NaN
4,HUMAN_OBSERVATION,2024-03-01T17:30:54,17:30:54-05:00,1.0,44.662766,-74.997231,23.0,Coleoptera,NaN,True,false,PNVYQ,Harmonia axyridis,NaN,NaN


In [ ]:
df['eventDate'] = pd.to_datetime(df['eventDate'], errors='coerce').dt.date

In [ ]:
min_date = df['eventDate'].dropna().min()
max_date = df['eventDate'].dropna().max()
print(f"Minimum event date: {min_date}")
print(f"Maximum event date: {max_date}")

Minimum event date: 2024-01-01
Maximum event date: 2024-12-31


In [11]:
species_counts = df['species'].value_counts()
species_to_keep = species_counts[species_counts >= 20].index
df = df[df['species'].isin(species_to_keep)]
print(f"DataFrame filtered to {len(species_to_keep)} species with at least 20 observations.")
print(f"New DataFrame shape: {df.shape}")

DataFrame filtered to 2936 species with at least 20 observations.
New DataFrame shape: (774869, 15)


In [ ]:
import requests
import time
import json

unique_species = df["species"].dropna().unique()
print(f"{len(unique_species)} unique species to look up")

common_name_map = {}
headers = {"User-Agent": "your-project-name (your-github-or-email)"}

for i, name in enumerate(unique_species):
    try:
        resp = requests.get(
            "https://api.inaturalist.org/v1/taxa",
            params={"q": name, "rank": "species", "per_page": 1},
            headers=headers
        )
        resp.raise_for_status()  
        results = resp.json().get("results", [])
        if results:
            common_name_map[name] = results[0].get("preferred_common_name") or name
        else:
            common_name_map[name] = name  
    except requests.exceptions.RequestException as e:
        print(f"Request failed for species '{name}': {e}")
        if resp is not None:
            print(f"Status Code: {resp.status_code}")
            print(f"Response Text: {resp.text[:500]}...") 
        common_name_map[name] = name 
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error for species '{name}': {e}")
        if resp is not None:
            print(f"Status Code: {resp.status_code}")
            print(f"Response Text: {resp.text[:500]}...") 
        common_name_map[name] = name 

    time.sleep(1)  
    if (i + 1) % 100 == 0: 
        print(f"Processed {i+1} species...")

# save it so you never have to repeat this lookup
with open("common_names.json", "w") as f:
    json.dump(common_name_map, f)

df["common_name"] = df["species"].map(common_name_map)


2936 unique species to look up
Processed 100 species...
Processed 200 species...
Processed 300 species...
Processed 400 species...
Processed 500 species...
Processed 600 species...
Processed 700 species...
Processed 800 species...
Processed 900 species...
Processed 1000 species...
Processed 1100 species...
Processed 1200 species...
Processed 1300 species...
Processed 1400 species...
Processed 1500 species...
Processed 1600 species...
Processed 1700 species...
Processed 1800 species...
Processed 1900 species...
Processed 2000 species...
Processed 2100 species...
Processed 2200 species...
Processed 2300 species...
Processed 2400 species...
Processed 2500 species...
Processed 2600 species...
Processed 2700 species...
Processed 2800 species...
Processed 2900 species...


In [14]:
df.head()

,basisOfRecord,eventDate,eventTime,day,decimalLatitude,decimalLongitude,coordinateUncertaintyInMeters,order,elevation,hasCoordinate,hasGeospatialIssues,speciesKey,species,isInvasive,iucnRedListCategory,common_name
0,HUMAN_OBSERVATION,2024-01-08,09:41:10-05:00,8.0,46.232197,-67.793869,34.0,Hemiptera,NaN,True,false,6PGH7,Leptoglossus phyllopus,NaN,NaN,Eastern Leaf-footed Bug
1,HUMAN_OBSERVATION,NaT,17:54:00-05:00,24.0,43.022518,-77.060684,15.0,Lepidoptera,NaN,True,false,943K8,Hypena scabra,NaN,NaN,Green Cloverworm Moth
2,HUMAN_OBSERVATION,NaT,08:50:00-05:00,9.0,42.450188,-71.120193,6.0,Coleoptera,NaN,True,false,8NBZS,Photinus corruscus,NaN,LC,Winter Firefly
4,HUMAN_OBSERVATION,2024-03-01,17:30:54-05:00,1.0,44.662766,-74.997231,23.0,Coleoptera,NaN,True,false,PNVYQ,Harmonia axyridis,NaN,NaN,Asian Lady Beetle
5,HUMAN_OBSERVATION,2024-03-14,13:56:54-04:00,14.0,43.083748,-71.976891,6.0,Lepidoptera,NaN,True,false,486GN,Nymphalis antiopa,NaN,LC,Mourning Cloak


In [15]:
# Save the filtered DataFrame to Google Drive
df.to_csv('/content/drive/MyDrive/filtered_occurrence_data.csv', index=False)
print("DataFrame saved to /content/drive/MyDrive/filtered_occurrence_data.csv")

DataFrame saved to /content/drive/MyDrive/filtered_occurrence_data.csv


In [18]:
df["common_name"].value_counts()


,count
common_name,
Common Eastern Bumble Bee,15951
Asian Lady Beetle,11027
Western Honey Bee,10713
Spotted Lanternfly,10299
Monarch,9312
...,...
Antherophagus ochraceus,20
Black Locust Leafminer,20
Orange-shouldered Leafminer,20


In [23]:
import pandas as pd
import numpy as np
df = pd.read_csv(
    "../data/FIXED_filtered_occurrence_data.csv", parse_dates=["eventDate"]
)

In [24]:
df = df.dropna(subset=["eventDate", "eventTime", "decimalLatitude", "decimalLongitude"])

df.shape

(769959, 16)

In [25]:
import geopandas as gpd
from shapely.geometry import Point


ecoregions = gpd.read_file("../data/us_eco_l3/us_eco_l3.shp").to_crs("EPSG:4326")

points = gpd.GeoDataFrame(
    df.copy(),
    geometry=[Point(xy) for xy in zip(df["decimalLongitude"], df["decimalLatitude"])],
    crs="EPSG:4326"
)

joined = gpd.sjoin(points, ecoregions[["geometry", "US_L3NAME"]], how="left", predicate="within")
df["ecoregion"] = joined["US_L3NAME"].values

missing = df["ecoregion"].isna()
if missing.any():
    nearest = gpd.sjoin_nearest(points[missing], ecoregions[["geometry", "US_L3NAME"]])
    df.loc[missing, "ecoregion"] = nearest["US_L3NAME"].values

print(df["ecoregion"].value_counts())

c:\Users\dothe\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


ecoregion
Northeastern Coastal Zone                          133194
Northern Piedmont                                   86524
Eastern Corn Belt Plains                            68259
Northeastern Highlands                              66210
Atlantic Coastal Pine Barrens                       42029
Southeastern Plains                                 41990
Ridge and Valley                                    38467
Western Allegheny Plateau                           38282
Middle Atlantic Coastal Plain                       36509
Erie Drift Plain                                    35810
Southern Michigan/Northern Indiana Drift Plains     32521
Eastern Great Lakes Lowlands                        26341
Piedmont                                            19938
Interior Plateau                                    19327
Huron/Erie Lake Plains                              16372
Northern Allegheny Plateau                          15599
Acadian Plains and Hills                            13489
Cent

In [30]:
import rasterio
from rasterio.warp import transform

with rasterio.open("../data/Annual_NLCD_LndCov_2024_CU_C1V1/Annual_NLCD_LndCov_2024_CU_C1V1.tif") as src:
    xs, ys = transform("EPSG:4326", src.crs, df["decimalLongitude"].tolist(), df["decimalLatitude"].tolist())
    land_cover_codes = [v[0] for v in src.sample(zip(xs, ys))]

df["land_cover_code"] = land_cover_codes


nlcd_labels = {
    11: "Open Water",
    12: "Perennial Ice/Snow",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    51: "Dwarf Scrub",       
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    72: "Sedge/Herbaceous",   
    73: "Lichens",            
    74: "Moss",               
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
}
df["land_cover_label"] = df["land_cover_code"].map(nlcd_labels)
print(df["land_cover_label"].value_counts())

land_cover_label
Developed, Open Space           146944
Deciduous Forest                139848
Developed, Low Intensity        137068
Developed, Medium Intensity      85093
Pasture/Hay                      64961
Mixed Forest                     43089
Woody Wetlands                   37838
Developed, High Intensity        24439
Open Water                       17305
Evergreen Forest                 17159
Cultivated Crops                 16608
Emergent Herbaceous Wetlands     11415
Barren Land                      10086
Grassland/Herbaceous              9318
Shrub/Scrub                       8422
Name: count, dtype: int64


In [31]:
df.to_csv("../data/occurrence_ecoregions.csv", index=False)

In [32]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/occurrence_ecoregions.csv")


In [ ]:
import xarray as xr

base_path = "../data/era5/"
quarters = ["jan-march", "april-jun", "jul-sep", "oct-dec"]

merged_quarters = []

for q in quarters:
    instant_file = f"{base_path}{q}_instant.nc"
    accum_file = f"{base_path}{q}_accum.nc"

    print(f"Loading and merging {q}...")
    instant_ds = xr.open_dataset(instant_file, engine="netcdf4")
    accum_ds = xr.open_dataset(accum_file, engine="netcdf4")

    quarter_merged = xr.merge([instant_ds, accum_ds])
    merged_quarters.append(quarter_merged)

print("Concatenating all quarters along valid_time...")
era5_full_year = xr.concat(merged_quarters, dim="valid_time")

print("\n--- Success! Full Year Dataset Combined ---")
print(era5_full_year)

In [ ]:
era5_full_year.to_netcdf("../data/era5/era5_full.nc", engine="netcdf4")
print("Saved successfully!")

Saved successfully!


In [34]:
import xarray as xr

era5_full = xr.open_dataset("../data/era5/era5_full.nc")
print(era5_full.dims)
print(list(era5_full.data_vars))
print(era5_full.coords)
print("longitude range:", era5_full.longitude.min().item(), "to", era5_full.longitude.max().item())


c:\Users\dothe\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xarray\backends\plugins.py:109: RuntimeWarning: Engine 'cfgrib' loading failed:
Cannot find the ecCodes library
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


FrozenMappingWarningOnValuesAccess({'valid_time': 8784, 'latitude': 49, 'longitude': 101})
['d2m', 't2m', 'tcc', 'sd', 'stl1', 'tp']
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 70kB 2024-01-01 ... 2024-12-31T23...
    expver      (valid_time) <U4 141kB ...
  * latitude    (latitude) float64 392B 48.0 47.75 47.5 ... 36.5 36.25 36.0
  * longitude   (longitude) float64 808B -91.0 -90.75 -90.5 ... -66.25 -66.0
    number      int64 8B ...
longitude range: -91.0 to -66.0


In [ ]:
points = xr.Dataset({
    "latitude": ("points", df["decimalLatitude"].astype("float64").values),
    "longitude": ("points", df["decimalLongitude"].astype("float64").values),
    "valid_time": ("points", df["eventDate"].values)
})

sampled = era5_full.sel(latitude=points.latitude, longitude=points.longitude,
                         valid_time=points.valid_time, method="nearest")

df["hourly_temp_C"] = sampled["t2m"].values - 273.15
df["dewpoint_C"] = sampled["d2m"].values - 273.15
df["soil_temp_C"] = sampled["stl1"].values - 273.15
df["cloud_cover_pct"] = sampled["tcc"].values * 100
df["snow_depth_m"] = sampled["sd"].values
df["precip_m"] = sampled["tp"].values
df["precip_m"] = sampled["tp"].values

def relative_humidity(t2m_c, d2m_c):
    e = 6.112 * np.exp((17.67 * d2m_c) / (d2m_c + 243.5))
    es = 6.112 * np.exp((17.67 * t2m_c) / (t2m_c + 243.5))
    return 100 * (e / es)

df["hourly_humidity_percent"] = relative_humidity(df["hourly_temp_C"], df["dewpoint_C"])


In [38]:
def precip_phase(row):
    if row["precip_m"] <= 0:
        return "none"
    elif row["hourly_temp_C"] < 0:
        return "snow"
    elif 0 <= row["hourly_temp_C"] < 2:
        return "mixed/freezing"  # the ambiguous near-freezing band
    else:
        return "rain"

df["precip_type"] = df.apply(precip_phase, axis=1)
print(df["precip_type"].value_counts())
df.head()

precip_type
none              527984
rain              240831
snow                 616
mixed/freezing       528
Name: count, dtype: int64


,basisOfRecord,eventDate,eventTime,day,decimalLatitude,decimalLongitude,coordinateUncertaintyInMeters,order,elevation,hasCoordinate,...,land_cover_code,land_cover_label,hourly_temp_C,dewpoint_C,soil_temp_C,cloud_cover_pct,snow_depth_m,precip_m,hourly_humidity_percent,precip_type
0,HUMAN_OBSERVATION,2024-01-08 09:41:10,09:41:10-05:00,8.0,46.232197,-67.793870,34.0,Hemiptera,NaN,True,...,22,"Developed, Low Intensity",-11.906769,-14.569794,-10.137787,96.188354,0.008718,0.000000,80.564407,none
1,HUMAN_OBSERVATION,2024-01-24 17:54:00,17:54:00-05:00,24.0,43.022520,-77.060684,15.0,Lepidoptera,NaN,True,...,21,"Developed, Open Space",2.642975,0.755096,-1.829926,100.000000,0.008189,0.000518,87.362221,rain
2,HUMAN_OBSERVATION,2024-02-09 08:50:00,08:50:00-05:00,9.0,42.450188,-71.120190,6.0,Coleoptera,NaN,True,...,41,Deciduous Forest,-1.682465,-2.708221,-0.514252,67.944336,0.000000,0.000000,92.700455,none
3,HUMAN_OBSERVATION,2024-03-01 17:30:54,17:30:54-05:00,1.0,44.662766,-74.997230,23.0,Coleoptera,NaN,True,...,23,"Developed, Medium Intensity",2.897858,-7.541443,-2.525726,60.974121,0.003399,0.000000,46.182762,none
4,HUMAN_OBSERVATION,2024-03-14 13:56:54,13:56:54-04:00,14.0,43.083748,-71.976890,6.0,Lepidoptera,NaN,True,...,21,"Developed, Open Space",5.096100,1.256012,0.639557,93.933105,0.000000,0.000000,76.219757,none


In [39]:
df.to_csv("../data/FULL_INSECT_DATA.csv", index=False)